# Step 3: Classify All PATSTAT Patents as ICE or EV

## Purpose
Assign each of the ~114 M PATSTAT applications a technology label based on
its Cooperative Patent Classification (CPC) codes:

- **ICE / Dirty** — codes covering internal combustion engine components,
  exhaust systems, fuel injection, etc.
- **EV / Green** — codes covering electric propulsion, battery management,
  charging infrastructure, hybrid drivetrains, etc.
- **Hybrid** — a sub-flag for the B60K 6/ group (multi-prime-mover
  arrangements), which straddles both categories.

The counts are used in Step 5 to decide whether each individual patent
application is primarily "Green" or "Dirty" (whichever has more matching
CPC codes wins).

## Inputs (raw PATSTAT files, all in `patstat_raw/` next to the repository folder; see SETUP.md, Part 2)
| File | Description |
|------|-------------|
| `patstat_raw/tls201_united_no_exc_columns.csv` | All 114 M PATSTAT applications (metadata) |
| `patstat_raw/tls224_part01/02/03.csv` | CPC classification codes for all applications |

## Outputs
| File | Description |
|------|-------------|
| `cpc_codes_aggregated.csv` | count_ICE / count_EV / count_Hybrid per appln_id (6.7 M rows with ≥1 relevant code) |
| `patents_cpc_classified.csv` | Full 114 M applications with CPC counts (0 where no relevant codes) — read by **Step 4** |


In [ ]:
import pandas as pd
import numpy as np
import gc
from pathlib import Path


In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
# Raw PATSTAT exports live outside the repository (see SETUP.md, Part 2).
# Default: a folder `patstat_raw/` next to the repository folder; set the
# environment variable PATSTAT_DIR to use another location.
import os
PATSTAT_DIR    = Path(os.environ.get('PATSTAT_DIR', '../../../patstat_raw'))
TLS201_FILE    = PATSTAT_DIR / 'tls201_united_no_exc_columns.csv'
TLS224_PARTS   = [PATSTAT_DIR / 'tls224_part01.csv',
                  PATSTAT_DIR / 'tls224_part02.csv',
                  PATSTAT_DIR / 'tls224_part03.csv']
_missing = [p for p in [TLS201_FILE] + TLS224_PARTS if not p.exists()]
if _missing:
    for p in _missing:
        print('MISSING', p)
    raise FileNotFoundError(
        f'{len(_missing)} raw PATSTAT file(s) missing. Put ALL PATSTAT exports into '
        f'{PATSTAT_DIR.resolve()} (default: patstat_raw/ next to the repository folder; '
        'see SETUP.md, Part 2).')
OUTPUT_CPC     = 'cpc_codes_aggregated.csv'     # intermediate aggregated counts
OUTPUT_MAIN    = 'patents_cpc_classified.csv'   # consumed by Step 4
CHUNK_SIZE     = 1_000_000   # rows per chunk when processing TLS224


In [ ]:
def get_ice_patterns() -> list:
    """
    Return the list of CPC regex patterns for Internal Combustion Engine (ICE)
    technology.

    Each pattern targets a specific IPC/CPC subclass or subgroup associated
    with combustion engine design, lubrication, exhaust treatment, fuel
    delivery, ignition, or thermodynamic recovery from ICE exhaust.

    Source references are listed in square brackets (citing the IPC/CPC
    classification schedules and the Directed Technical Change literature).
    """
    return [
        # ── Engine structure ──────────────────────────────────────────────────
        r'^F01B',         # Reciprocating-piston machines (general ICE structure)
        r'^F01L',         # Cyclically operating valves (valvetrain)
        r'^F01M',         # Lubrication of ICEs
        r'^F01N',         # Gas-flow silencers / exhaust treatment for ICEs
        r'^F01P',         # Cooling of ICEs
        r'^H01T',         # Sparking plugs (ICE ignition)
        r'^H02J\s*7/14',  # Charging batteries from engine-driven generators
        r'^F02B',         # Combustion piston engines; general combustion engines
        r'^F02D',         # Controlling combustion engines
        r'^F02F',         # Cylinders, pistons, casings, sealings for ICEs
        r'^F02G\s*5/',    # Profiting from waste heat of combustion engines
        r'^F02M',         # Supplying combustible mixtures (carburettors, injection)
        r'^F02N',         # Starting of combustion engines
        r'^F02P',         # Ignition (other than compression ignition)
        # ── Pumps / turbines in ICE context ───────────────────────────────────
        r'^F04B\s*17/05',      r'^F04B\s*35/002',    r'^F04B\s*41/04',
        r'^F04C\s*15/062',     r'^F04C\s*29/122',
        r'^F05B\s*2203/06',    r'^F05D\s*2220/40',
        r'^F15B\s*2211/20523',
        # ── Crankshafts, pistons, connecting-rods ────────────────────────────
        r'^F16C\s*3/04',  r'^F16C\s*3/06',  r'^F16C\s*3/14',
        r'^F16C\s*3/16',  r'^F16C\s*3/20',  r'^F16C\s*3/22',
        r'^F16C\s*5/00',  r'^F16C\s*7/00',  r'^F16C\s*9/00',
        r'^F16F\s*15/24', r'^F16F\s*15/26', r'^F16F\s*15/283',
        r'^F16H\s*57/0475', r'^F16H\s*59/74', r'^F16H\s*61/21',
        r'^F16H\s*2059/74', r'^F16H\s*2063/50', r'^F16H\s*2500/1062',
        r'^F16H\s*2500/1068',
        r'^F16J\s*1/00',  r'^F16J\s*7/00',  r'^F16J\s*9/00', r'^F16J\s*10/00',
        r'^F16M\s*1/021', r'^F16M\s*5/00',
        r'^F16N\s*39/08',
        # ── Glow plugs, heat exchangers for ICE ──────────────────────────────
        r'^F23Q\s*7/001', r'^F23Q\s*2007/',
        r'^F25B\s*2327/001',
        r'^F28D\s*11/025',    r'^F28D\s*2021/0026', r'^F28D\s*2021/0082',
        r'^F28D\s*2021/0087', r'^F28D\s*2021/0089', r'^F28D\s*2021/0091',
        r'^F28D\s*2021/0094',
        # ── Vehicle ICE integration (B60K) ────────────────────────────────────
        r'^B60K\s*5/',    # Arrangement/mounting of ICE or jet units in vehicle
        r'^B60K\s*13/',   # Combustion air intake / gas exhaust in vehicle
        r'^B60K\s*15/',   # Fuel-tank arrangement (ICE fuel supply)
        r'^B60H\s*1/025', r'^B60H\s*1/18', r'^B60H\s*1/22',
        r'^B60Q\s*1/2688',
    ]


def get_ev_patterns() -> list:
    """
    Return the list of CPC regex patterns for Electric Vehicle (EV) and
    Hybrid technology.

    Covers electric propulsion, battery management systems (BMS), charging
    infrastructure, regenerative braking, power electronics for EVs, and
    hybrid drivetrain arrangements (B60K 6/).
    """
    return [
        # ── Battery and energy storage ────────────────────────────────────────
        r'^H01M\s*2220/20',   # Batteries for motive / vehicle systems
        # ── Electric motors in vehicle context ───────────────────────────────
        r'^H02K\s*7/006',     # Motors structurally associated with vehicle drivetrain
        # ── Charging infrastructure ───────────────────────────────────────────
        r'^H02J\s*7/00',      # Circuit arrangements for charging (grid charging)
        r'^H02J\s*2207/20',   # Charging via power electronics converter
        r'^H02J\s*50/',       # Wireless power transfer (inductive EV charging)
        # ── Hybrid cooling / auxiliary power ─────────────────────────────────
        r'^F01P\s*2050/24',   # Cooling specifically for hybrid vehicles
        r'^F01P\s*2050/30',   # Cooling of circuit boards (inverters, BMS)
        r'^F03D\s*9/32',      # Wind motors on moving vehicles (auxiliary charging)
        r'^F03G\s*7/081',     # Energy recovery from road/rail vibrations
        # ── Electric drivetrains in gearbox indexing ─────────────────────────
        r'^F16D\s*2500/1064', r'^F16D\s*2500/1066',
        r'^F16H\s*3/724',     r'^F16H\s*3/727',
        r'^F16H\s*57/02034',  r'^F16H\s*57/0476',
        r'^F16H\s*2061/6602', r'^F16H\s*2200/0021',
        # ── Fuel-cell heat exchangers ─────────────────────────────────────────
        r'^F25B\s*2240/10',   r'^F28D\s*2021/0043',
        # ── EV propulsion (B60L covers ALL electric propulsion subgroups) ─────
        r'^B60L',             # All subgroups: charging, traction motors, monitoring
        # ── Vehicle EV integration (B60K 1, B60K 6) ──────────────────────────
        r'^B60K\s*1/',        # Mounting of electric propulsion units in vehicle
        r'^B60K\s*6/',        # Plural diverse prime-movers (hybrid arrangements)
        # ── Hybrid control systems (B60W) ─────────────────────────────────────
        r'^B60W\s*20/',       # Control systems for hybrid vehicles
        r'^B60W\s*10/08',     # Conjoint control including electric propulsion
        r'^B60W\s*10/24',     # Conjoint control of energy storage means
        r'^B60W\s*10/26',     # Conjoint control of electrical energy storage
        r'^B60W\s*10/28',     # Conjoint control including fuel cells
        # ── Thermal management for EV battery ────────────────────────────────
        r'^B60H\s*1/00278',   # HVAC for EV battery pack
        r'^B60H\s*1/143',     # Heat from cooling an electric component
    ]


In [ ]:
def process_cpc_chunk(filepath: str,
                      ice_patterns: list,
                      ev_patterns: list) -> pd.DataFrame:
    """
    Load one TLS224 file and tag each CPC entry as ICE, EV, or Hybrid.

    Strategy:
      1. Build a combined regex from all ICE + EV patterns to filter the file
         down to only relevant rows (reduces ~114 M rows to ~6.7 M).
      2. Tag each surviving row with is_ICE, is_EV, and is_Hybrid flags.
         is_Hybrid marks the B60K 6/ subgroup, which represents vehicles
         with *both* combustion and electric powertrains (ambiguous category).

    Parameters
    ----------
    filepath     : str  — path to tls224_partNN.csv
    ice_patterns : list — from get_ice_patterns()
    ev_patterns  : list — from get_ev_patterns()

    Returns
    -------
    pd.DataFrame : filtered and tagged CPC entries
    """
    print(f'  Processing {Path(filepath).name}...', end=' ', flush=True)

    # Combined filter — keeps only rows that belong to *any* target category
    combined_regex = '|'.join(ice_patterns + ev_patterns)

    df = pd.read_csv(filepath, dtype={'appln_id': 'int32'})
    mask = df['cpc_class_symbol'].str.contains(combined_regex, regex=True, na=False)
    df  = df[mask].copy()

    # Tag category membership for each surviving CPC entry
    ice_re  = '|'.join(ice_patterns)
    ev_re   = '|'.join(ev_patterns)
    hyb_re  = r'^B60K\\s*6/'   # hybrid-specific subgroup

    df['is_ICE']    = df['cpc_class_symbol'].str.contains(ice_re,  regex=True).astype('int8')
    df['is_EV']     = df['cpc_class_symbol'].str.contains(ev_re,   regex=True).astype('int8')
    df['is_Hybrid'] = df['cpc_class_symbol'].str.contains(hyb_re,  regex=True).astype('int8')

    print(f'{len(df):,} relevant CPC entries')
    return df


In [ ]:
def aggregate_cpc_codes(df_cpc: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate CPC flags to one row per patent application (appln_id).

    After process_cpc_chunk(), a single application may have multiple rows
    (one per matching CPC code).  We collapse these to a single row by
    summing the flag columns, giving us *counts* of how many ICE, EV, and
    Hybrid codes each application carries.

    These counts are later used in the classification rule in Step 5:
      Green  →  count_ICE ≤  count_EV + count_Hybrid
      Dirty  →  count_ICE >  count_EV + count_Hybrid

    Parameters
    ----------
    df_cpc : pd.DataFrame — concatenated output of process_cpc_chunk()

    Returns
    -------
    pd.DataFrame : (appln_id, count_ICE_codes, count_EV_codes, count_Hybrid_codes)
    """
    agg = (
        df_cpc.groupby('appln_id')
        .agg(count_ICE_codes=('is_ICE', 'sum'),
             count_EV_codes=('is_EV',  'sum'),
             count_Hybrid_codes=('is_Hybrid', 'sum'))
        .reset_index()
    )
    print(f'Aggregated to {len(agg):,} unique patent applications with ≥1 relevant CPC code')
    return agg


In [ ]:
def load_all_applications(filepath: str) -> pd.DataFrame:
    """
    Load the full PATSTAT applications table (TLS201, ~114 M rows).

    We keep only the columns needed for the subsequent merge and analysis:
    application ID, authority, filing date, IPR type, and patent-family ID.

    Parameters
    ----------
    filepath : str  — path to tls201_united_no_exc_columns.csv

    Returns
    -------
    pd.DataFrame : all 114 M applications
    """
    print(f'Loading TLS201 ({filepath})... ', end='', flush=True)
    df = pd.read_csv(filepath)
    print(f'{len(df):,} rows loaded')
    return df


def classify_all_patents(tls201: pd.DataFrame,
                         cpc_aggregated: pd.DataFrame) -> pd.DataFrame:
    """
    Left-join TLS201 with the aggregated CPC code counts.

    A left join keeps all 114 M applications; those with no relevant CPC
    codes receive NaN in the count columns, which are then filled with 0.
    This is important for the panel-balancing step in Step 5: if a firm-year
    cell has zero relevant patents, it must appear explicitly (as 0) rather
    than being absent.

    Parameters
    ----------
    tls201         : pd.DataFrame — full applications table
    cpc_aggregated : pd.DataFrame — from aggregate_cpc_codes()

    Returns
    -------
    pd.DataFrame : 114 M rows with count_ICE_codes, count_EV_codes, count_Hybrid_codes
    """
    merged = pd.merge(tls201, cpc_aggregated, on='appln_id', how='left')

    count_cols = ['count_ICE_codes', 'count_EV_codes', 'count_Hybrid_codes']
    merged[count_cols] = merged[count_cols].fillna(0).astype('int16')

    print(f'Apps with ≥1 ICE code : {(merged["count_ICE_codes"] > 0).sum():,}')
    print(f'Apps with ≥1 EV code  : {(merged["count_EV_codes"]  > 0).sum():,}')
    return merged


In [ ]:
def main() -> None:
    """
    Full Step 3 pipeline:
      1. Load ICE and EV CPC patterns.
      2. Process each TLS224 chunk: filter → tag → concatenate.
      3. Aggregate to one row per application with CPC code counts.
      4. Save the aggregated counts (cpc_codes_aggregated.csv).
      5. Left-join onto all 114 M applications and save patents_cpc_classified.csv.
    """
    print('=== Step 3: CPC Technology Classification ===')

    ice_patterns = get_ice_patterns()
    ev_patterns  = get_ev_patterns()
    print(f'ICE patterns: {len(ice_patterns)}  |  EV patterns: {len(ev_patterns)}')

    # Process each TLS224 part
    print('\nProcessing TLS224 CPC files...')
    parts = [process_cpc_chunk(fp, ice_patterns, ev_patterns) for fp in TLS224_PARTS]
    df_cpc = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()

    # Aggregate and save intermediate
    cpc_agg = aggregate_cpc_codes(df_cpc)
    del df_cpc
    gc.collect()
    cpc_agg.to_csv(OUTPUT_CPC, index=False)
    print(f'Saved → {OUTPUT_CPC}')

    # Load all applications and classify
    print('\nLoading full applications table...')
    tls201 = load_all_applications(TLS201_FILE)

    classified = classify_all_patents(tls201, cpc_agg)
    del tls201, cpc_agg
    gc.collect()

    classified.to_csv(OUTPUT_MAIN, index=False)
    print(f'\n✓ Saved → {OUTPUT_MAIN}  ({len(classified):,} rows)')


main()
